In [1]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import json

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print(" Libraries imported successfully!")

 Libraries imported successfully!


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

valid_cohorts = [
    "Participants 05-08",
    "Participants 09-12",
    "Participants 17-20",
    "Participants 29-32",
    "Participants 45-48",
    "Participants 53-56",
    "Participants 57-60",
    "Participants 61-64",
    "Participants 65-68",
    "Participants 69-72",
    "Participants 77-80",
    "Participants 85-88",
    "Participants 97-100",
    "Participants 101-104",
]

base_path = Path("/Volumes/Klimt/Studies/Collaborative Kitchen Cognition (KITCO)/KITCO Study C/Food Preparation Study/4. Raw Data")

pathsToProcess = []
for participantPath in base_path.glob("Participants*"):
    if participantPath.name not in valid_cohorts:
        continue
    for kitcoPath in participantPath.iterdir():
        if kitcoPath.name.startswith('.') or not kitcoPath.is_dir():
            continue
        for subPath in kitcoPath.iterdir():
            if subPath.name.startswith('.') or not subPath.is_dir():
                continue
            # Check if this looks like a timestamp folder (starts with 20)
            if subPath.name.startswith('20'):
                pathsToProcess.append((participantPath.name, subPath))
            else:
                # Go one level deeper for cases like KITCO3FEB13no2a
                for dataTimeFolder in subPath.iterdir():
                    if dataTimeFolder.name.startswith('.') or not dataTimeFolder.is_dir():
                        continue
                    if dataTimeFolder.name.startswith('20'):
                        pathsToProcess.append((participantPath.name, dataTimeFolder))

filteredPaths = []
skipped_sessions = {}

def log_skip(cohort, session, reason):
    if cohort not in skipped_sessions:
        skipped_sessions[cohort] = []
    skipped_sessions[cohort].append(f"  Session {session}: {reason}")

for cohort_name, path in pathsToProcess:
    numDirs = sum(1 for d in path.iterdir() if d.is_dir() and not d.name.startswith('.'))
    if numDirs != 2:
        if numDirs > 0:  # only log if there's something there but wrong count
            log_skip(cohort_name, path.name, f"expected 2 sensor dirs, found {numDirs}")
        continue  # silently skip empty folders
    filteredPaths.append((cohort_name, path))

print(f"Found {len(filteredPaths)} valid sessions")
for cohort, path in sorted(filteredPaths):
    print(f"  {cohort} | {path.name}")

Found 52 valid sessions
  Participants 05-08 | 2025_11_28-22_39_22
  Participants 05-08 | 2025_11_28-23_07_45
  Participants 05-08 | 2025_11_28-22_42_29
  Participants 05-08 | 2025_11_28-22_42_49
  Participants 05-08 | 2025_11_28-23_04_26
  Participants 05-08 | 2025_11_28-23_07_21
  Participants 09-12 | 2025_12_01-22_37_05
  Participants 09-12 | 2025_12_01-23_08_47
  Participants 09-12 | 2025_12_01-22_31_58
  Participants 09-12 | 2025_12_01-23_08_42
  Participants 17-20 | 2025_12_16-22_00_52
  Participants 17-20 | 2025_12_16-22_30_26
  Participants 17-20 | 2025_12_16-22_32_49
  Participants 17-20 | 2025_12_16-22_01_30
  Participants 17-20 | 2025_12_16-22_29_41
  Participants 17-20 | 2025_12_16-22_29_50
  Participants 29-32 | 2025_12_19-19_20_59
  Participants 29-32 | 2025_12_19-19_41_02
  Participants 29-32 | 2025_12_19-19_21_15
  Participants 29-32 | 2025_12_19-19_42_49
  Participants 45-48 | 2026_02_09-23_59_21
  Participants 45-48 | 2026_02_10-00_17_15
  Participants 45-48 | 2026_02

In [3]:
def load_hr_data(sensor_id, data_directory):
    sensor_dir = data_directory / sensor_id
    hr_file = list(sensor_dir.glob("HR-*.csv"))[0]
    
    with open(hr_file, 'r') as f:
        header = f.readline().strip()
    
    hr_sampling_rate = int(header[-1])
    df = pd.read_csv(hr_file, skiprows=1)
    
    print(f"✓ Loaded HR data for sensor {sensor_id}")
    print(f"  File: {hr_file.name}")
    print(f"  Total samples: {len(df):,}")
    print(f"  HR range: {df['Average'].min():.1f} - {df['Average'].max():.1f} BPM")
    print(f"  RR interval range: {df['RRData'].min():.0f} - {df['RRData'].max():.0f} ms")
    print(f"  Sampling rate (from header): {hr_sampling_rate} Hz")
    
    return df, hr_sampling_rate

In [4]:
# Extract experiment window timestamps
def get_experiment_window(annotations_df):
    video_start = annotations_df[annotations_df['Type'] == 'VideoAnnotationStart']['Timestamp']
    video_stop = annotations_df[annotations_df['Type'] == 'VideoAnnotationStop']['Timestamp']
    
    if not video_start.empty and not video_stop.empty:
        start = video_start.values[0]
        end = video_stop.values[0]
        annotation_type = "Video"
    else:
        audio_start = annotations_df[annotations_df['Type'] == 'AudioAnnotationStart']['Timestamp']
        audio_stop = annotations_df[annotations_df['Type'] == 'AudioAnnotationStop']['Timestamp']
        
        if audio_start.empty or audio_stop.empty:
            print(f"⚠️ No valid annotation markers found. Available types:")
            print(f"   {annotations_df['Type'].unique().tolist()}")
            return None, None
        
        start = audio_start.values[0]
        end = audio_stop.values[0]
        annotation_type = "Audio"
    
    duration = end - start
    print(f"✓ Experiment window ({annotation_type}):")
    print(f"  Start: {start:.2f}")
    print(f"  End:   {end:.2f}")
    print(f"  Duration: {duration:.2f} seconds ({duration/60:.2f} minutes)")
    
    return start, end

In [5]:
def plot_ecg_overview(ecg_df1, ecg_df2, participant_names, sr1, sr2, window_seconds=10):
    """
    Plot a window of ECG data from both participants for visual inspection.
    
    Parameters:
    - window_seconds: Number of seconds to display (default: 10)
    """
    if len(ecg_df1) == 0 or len(ecg_df2) == 0:
        print("❌ One or both trimmed dataframes are empty — skipping plot.")
        return

    # Calculate number of samples for the window
    window_samples1 = int(window_seconds * sr1)
    window_samples2 = int(window_seconds * sr2)

    # Start from beginning instead of midpoint to avoid running off the end
    window1 = ecg_df1.iloc[:window_samples1]
    window2 = ecg_df2.iloc[:window_samples2]

    if len(window1) == 0 or len(window2) == 0:
        print("❌ Window slice is empty — recording may be shorter than window_seconds.")
        return

    # Create relative time in seconds
    time1 = window1['Timestamp'].values - window1['Timestamp'].values[0]
    time2 = window2['Timestamp'].values - window2['Timestamp'].values[0]

    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

    axes[0].plot(time1, window1['Sample'], linewidth=0.5, color='#2E86AB')
    axes[0].set_ylabel('ECG Amplitude', fontsize=12)
    axes[0].set_title(f'{participant_names[0]} - Raw ECG Signal', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(time2, window2['Sample'], linewidth=0.5, color='#A23B72')
    axes[1].set_xlabel('Time (seconds)', fontsize=12)
    axes[1].set_ylabel('ECG Amplitude', fontsize=12)
    axes[1].set_title(f'{participant_names[1]} - Raw ECG Signal', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [6]:
def load_ecg_data(sensor_id, data_directory):
    sensor_dir = data_directory / sensor_id
    ecg_file = list(sensor_dir.glob("ECG-*.csv"))[0]
    
    with open(ecg_file, 'r') as f:
        header = f.readline().strip()
    
    sampling_rate = int(header[-3:])
    df = pd.read_csv(ecg_file, skiprows=1)
    
    print(f"✓ Loaded ECG for sensor {sensor_id}")
    print(f"  File: {ecg_file.name}")
    print(f"  Total samples: {len(df):,}")
    print(f"  Time range: {df['Timestamp'].min():.2f} to {df['Timestamp'].max():.2f}")
    print(f"  Duration: {(df['Timestamp'].max() - df['Timestamp'].min()):.2f} seconds")
    print(f"  Sampling rate (from header): {sampling_rate} Hz")
    
    return df, sampling_rate

def trim_to_window(df, start_time, end_time):
    mask = (df['Timestamp'] >= start_time) & (df['Timestamp'] <= end_time)
    trimmed = df[mask].copy()
    trimmed.reset_index(drop=True, inplace=True)
    print(f"  Original samples: {len(df):,}")
    print(f"  Trimmed samples:  {len(trimmed):,}")
    print(f"  Kept: {len(trimmed)/len(df)*100:.1f}%")
    return trimmed

def load_hr_data(sensor_id, data_directory):
    sensor_dir = data_directory / sensor_id
    hr_file = list(sensor_dir.glob("HR-*.csv"))[0]
    
    with open(hr_file, 'r') as f:
        header = f.readline().strip()
    
    hr_sampling_rate = int(header[-1])
    df = pd.read_csv(hr_file, skiprows=1)
    
    print(f"✓ Loaded HR data for sensor {sensor_id}")
    print(f"  File: {hr_file.name}")
    print(f"  Total samples: {len(df):,}")
    print(f"  HR range: {df['Average'].min():.1f} - {df['Average'].max():.1f} BPM")
    print(f"  RR interval range: {df['RRData'].min():.0f} - {df['RRData'].max():.0f} ms")
    print(f"  Sampling rate (from header): {hr_sampling_rate} Hz")
    
    return df, hr_sampling_rate

def get_experiment_window(annotations_df):
    video_start = annotations_df[annotations_df['Type'] == 'VideoAnnotationStart']['Timestamp']
    video_stop = annotations_df[annotations_df['Type'] == 'VideoAnnotationStop']['Timestamp']
    
    if not video_start.empty and not video_stop.empty:
        start = video_start.values[0]
        end = video_stop.values[0]
        annotation_type = "Video"
    else:
        audio_start = annotations_df[annotations_df['Type'] == 'AudioAnnotationStart']['Timestamp']
        audio_stop = annotations_df[annotations_df['Type'] == 'AudioAnnotationStop']['Timestamp']
        
        if audio_start.empty or audio_stop.empty:
            print(f"⚠️ No valid annotation markers found. Available types:")
            print(f"   {annotations_df['Type'].unique().tolist()}")
            return None, None
        
        start = audio_start.values[0]
        end = audio_stop.values[0]
        annotation_type = "Audio"
    
    duration = end - start
    print(f"✓ Experiment window ({annotation_type}):")
    print(f"  Start: {start:.2f}")
    print(f"  End:   {end:.2f}")
    print(f"  Duration: {duration:.2f} seconds ({duration/60:.2f} minutes)")
    
    return start, end

print("✓ All functions loaded")

✓ All functions loaded


In [7]:
for cohort_name, finalPath in sorted(filteredPaths):
    data_dir = finalPath
    sensorNums = []
    for sensorPath in finalPath.iterdir():
        if '.csv' in sensorPath.name or '.mov' in sensorPath.name or sensorPath.name.startswith('.'):
            continue
        sensorNums.append(sensorPath.name)

    if len(sensorNums) != 2:
        log_skip(cohort_name, finalPath.name, f"expected 2 sensors, found {len(sensorNums)}: {sensorNums}")
        continue

    print(f"\n{'='*60}")
    print(f"Cohort: {cohort_name} | Session: {finalPath.name}")
    print(f"{'='*60}")

    annotation_files = list(data_dir.glob("Annotations-*.csv"))
    if not annotation_files:
        log_skip(cohort_name, finalPath.name, "no annotations file found")
        continue
    if annotation_files[0].stat().st_size == 0:
        log_skip(cohort_name, finalPath.name, "annotations file is empty")
        continue

    annotations = pd.read_csv(annotation_files[0])
    exp_start, exp_end = get_experiment_window(annotations)
    if exp_start is None:
        log_skip(cohort_name, finalPath.name, "no valid annotation window found")
        continue

    participant_mapping = {
        sensorNums[0]: "Participant_A",
        sensorNums[1]: "Participant_B"
    }

    ecg_p1, sr_p1 = load_ecg_data(sensorNums[0], data_dir)
    print()
    ecg_p2, sr_p2 = load_ecg_data(sensorNums[1], data_dir)

    print()
    hr_p1, hr_sr_p1 = load_hr_data(sensorNums[0], data_dir)
    print()
    hr_p2, hr_sr_p2 = load_hr_data(sensorNums[1], data_dir)

    print("\nTrimming Participant 1 (sensor {}):".format(sensorNums[0]))
    ecg_p1_trimmed = trim_to_window(ecg_p1, exp_start, exp_end)
    print("\nTrimming Participant 2 (sensor {}):".format(sensorNums[1]))
    ecg_p2_trimmed = trim_to_window(ecg_p2, exp_start, exp_end)

    if ecg_p1_trimmed.shape[0] == 0 or ecg_p2_trimmed.shape[0] == 0:
        log_skip(cohort_name, finalPath.name, "trimmed ECG dataframe is empty")
        continue

    print(f"\n{'='*60}")
    print("DATA LOADING SUMMARY")
    print(f"{'='*60}")
    print(f"\n📁 Cohort: {cohort_name} | Session: {finalPath.name}")
    print(f"⏱️  Duration: {(exp_end - exp_start):.1f} seconds ({(exp_end - exp_start)/60:.1f} minutes)")
    print(f"\n👥 Participant 1 ({participant_mapping[sensorNums[0]]}):")
    print(f"   Sensor ID: {sensorNums[0]}")
    print(f"   ECG samples: {len(ecg_p1_trimmed):,}")
    print(f"   HR samples:  {len(hr_p1):,}")
    print(f"   ECG sampling rate: {sr_p1:.0f} Hz")
    print(f"\n👥 Participant 2 ({participant_mapping[sensorNums[1]]}):")
    print(f"   Sensor ID: {sensorNums[1]}")
    print(f"   ECG samples: {len(ecg_p2_trimmed):,}")
    print(f"   HR samples:  {len(hr_p2):,}")
    print(f"   ECG sampling rate: {sr_p2:.0f} Hz")
    print("\n✅ Data successfully loaded and ready for analysis!")

    output_dir = Path("/Users/rachelsong/Desktop/Movesense/processed_data") / finalPath.name
    output_dir.mkdir(parents=True, exist_ok=True)

    ecg_p1_trimmed.to_csv(output_dir / f"ecg_{participant_mapping[sensorNums[0]]}.csv", index=False)
    ecg_p2_trimmed.to_csv(output_dir / f"ecg_{participant_mapping[sensorNums[1]]}.csv", index=False)
    hr_p1.to_csv(output_dir / f"hr_{participant_mapping[sensorNums[0]]}.csv", index=False)
    hr_p2.to_csv(output_dir / f"hr_{participant_mapping[sensorNums[1]]}.csv", index=False)

    metadata = {
        'session': finalPath.name,
        'cohort': cohort_name,
        'exp_start': exp_start,
        'exp_end': exp_end,
        'duration_seconds': exp_end - exp_start,
        'sampling_rate_p1': sr_p1,
        'sampling_rate_p2': sr_p2,
        'hr_sampling_rate_p1': hr_sr_p1,
        'hr_sampling_rate_p2': hr_sr_p2,
        'sensor_1': sensorNums[0],
        'sensor_2': sensorNums[1],
        'participant_1': participant_mapping[sensorNums[0]],
        'participant_2': participant_mapping[sensorNums[1]]
    }
    with open(output_dir / 'metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"\n✓ Data saved to {output_dir}/")
    for file in output_dir.glob("*"):
        print(f"    - {file.name}")

# Final report
if skipped_sessions:
    print(f"\n{'='*60}")
    print("⚠️  THE FOLLOWING COHORTS HAD SESSIONS THAT COULD NOT BE ANALYZED:")
    print(f"{'='*60}")
    for cohort, reasons in sorted(skipped_sessions.items()):
        print(f"\n{cohort}:")
        for r in reasons:
            print(r)
else:
    print("\n✅ All sessions processed successfully!")


Cohort: Participants 05-08 | Session: 2025_11_28-22_39_22
✓ Experiment window (Video):
  Start: 1764369933.38
  End:   1764370327.97
  Duration: 394.60 seconds (6.58 minutes)
✓ Loaded ECG for sensor 251430002239
  File: ECG-2025_11_28-22_39_22.csv
  Total samples: 165,232
  Time range: 1764369565.09 to 1764370853.17
  Duration: 1288.08 seconds
  Sampling rate (from header): 128 Hz

✓ Loaded ECG for sensor 251430002238
  File: ECG-2025_11_28-22_39_22.csv
  Total samples: 165,504
  Time range: 1764369562.89 to 1764370850.12
  Duration: 1287.23 seconds
  Sampling rate (from header): 128 Hz

✓ Loaded HR data for sensor 251430002239
  File: HR-2025_11_28-22_39_22.csv
  Total samples: 1,964
  HR range: 78.4 - 171.8 BPM
  RR interval range: 257 - 914 ms
  Sampling rate (from header): 1 Hz

✓ Loaded HR data for sensor 251430002238
  File: HR-2025_11_28-22_39_22.csv
  Total samples: 1,997
  HR range: 83.3 - 160.0 BPM
  RR interval range: 375 - 6367 ms
  Sampling rate (from header): 1 Hz

Trimm

### Key Concepts:

- **Unix Timestamp**: Seconds since January 1, 1970 (with microsecond precision)
- **Recording Start/Stop**: When the sensor system was turned on/off
- **Video/Audio Annotation Start/Stop**: The actual experimental period we want to analyze

### What is ECG?

**Electrocardiogram (ECG or EKG)** measures the electrical activity of your heart. Each heartbeat produces a characteristic waveform with distinct components:

```
       R
       |
   P   |   T
   /\  |  /\
  /  \ | /  \
 /    \|/    \
-------+-------
      Q S
```

- **P wave**: Atrial depolarization (atria contract)
- **QRS complex**: Ventricular depolarization (ventricles contract) - this is the heartbeat!
- **T wave**: Ventricular repolarization (ventricles relax)

The **R-peak** is the most prominent feature and is used to detect heartbeats.

### Heart Rate (HR) vs Inter-Beat Interval (IBI)

- **Heart Rate (HR)**: Number of beats per minute (BPM)
- **Inter-Beat Interval (IBI)**: Time between consecutive heartbeats (milliseconds)
- **RR Interval**: Same as IBI, named after the R-peaks in the ECG

These are inversely related: HR = 60,000 / IBI (when IBI is in milliseconds)